In [ ]:
# Mount Google Drive (if running in Google Colab)
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

# Install required packages (LightGBM, Optuna, etc.)
!pip install -q lightgbm optuna joblib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 6.7 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Modeling and evaluation
import lightgbm as lgb
import optuna
import joblib
from sklearn.model_selection import train_test_split, KFold, cross_val_score

import matplotlib.pyplot as plt

# Set a random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [ ]:
# Load the data from Google Drive

# Data availability:
# The aggregated domain rating dataset from Lin et al. [7] is publicly available via the Open Science Framework (OSF) repository (DOI: 10.17605/osf.io/9jwzs).
# Additional feature data are available from the corresponding author upon request.
# Due to licensing restrictions (specifically Moz domain metrics), the full compiled dataset is available only upon reasonable request.

# Handling of missing values: We restricted our analysis to domains with active URLs and complete Moz/Open PageRank metrics, which minimizes the incidence of missing values.

# Normalization: Tree‑based models such as LightGBM are insensitive to monotonic transformations of the input features. Consequently we did not apply normalization or scaling, as it would not improve performance but could reduce interpretability.

# Feature selection: We compiled all available link‑based and authority metrics from Moz and from Open PageRank. This resulted in 48 candidate features. Rather than applying a dimensionality‑reduction technique that might discard informative attributes, we retained the complete set and relied on LightGBM’s ensemble‑tree structure to weigh the relative importance of each predictor.


In [ ]:

# Define input feature columns (predictors) and target column
input_features = [
    "page_rank_decimal", "rank", "pages_to_page", "nofollow_pages_to_page",
    "redirect_pages_to_page", "external_pages_to_page", "external_nofollow_pages_to_page",
    "external_redirect_pages_to_page", "deleted_pages_to_page", "root_domains_to_page",
    "indirect_root_domains_to_page", "deleted_root_domains_to_page",
    "nofollow_root_domains_to_page", "pages_to_subdomain", "nofollow_pages_to_subdomain",
    "redirect_pages_to_subdomain", "external_pages_to_subdomain",
    "external_nofollow_pages_to_subdomain", "external_redirect_pages_to_subdomain",
    "deleted_pages_to_subdomain", "root_domains_to_subdomain", "deleted_root_domains_to_subdomain",
    "nofollow_root_domains_to_subdomain", "pages_to_root_domain", "nofollow_pages_to_root_domain",
    "redirect_pages_to_root_domain", "external_pages_to_root_domain",
    "external_indirect_pages_to_root_domain", "external_nofollow_pages_to_root_domain",
    "external_redirect_pages_to_root_domain", "deleted_pages_to_root_domain",
    "root_domains_to_root_domain", "indirect_root_domains_to_root_domain",
    "deleted_root_domains_to_root_domain", "nofollow_root_domains_to_root_domain",
    "page_authority", "domain_authority", "link_propensity", "spam_score",
    "root_domains_from_page", "nofollow_root_domains_from_page", "pages_from_page",
    "nofollow_pages_from_page", "root_domains_from_root_domain",
    "nofollow_root_domains_from_root_domain", "pages_from_root_domain",
    "nofollow_pages_from_root_domain", "pages_crawled_from_root_domain"
]
TARGET_COL = 'pc1'


In [ ]:


# Create the final modeling DataFrame with features and target
model_data = data[input_features + [TARGET_COL]].copy()

# Safety check: Ensure we have the expected structure
if TARGET_COL not in model_data.columns or len(input_features) != model_data.shape[1] - 1:
    raise RuntimeError("✘ Dataset not loaded properly or feature count mismatch. Please check input_features and target.")
print("Final model_data shape:", model_data.shape)


In [ ]:
# Adaptive stratification for regression target:
def build_strat_labels(y: pd.Series, n_bins: int, min_per_bin: int = 3):
    """
    Bin continuous target `y` into `n_bins` quantile buckets.
    Returns an array of bin labels (or None if infeasible).
    """
    valid = ~y.isna()
    try:
        labels = pd.qcut(y[valid], q=n_bins, labels=False, duplicates='drop')
    except Exception:
        return None
    # If any bin has fewer than min_per_bin samples or only one bin present, fail
    if labels.nunique() < 2 or (labels.value_counts().min() < min_per_bin):
        return None
    full_labels = pd.Series(-1, index=y.index, dtype=int)
    full_labels.loc[valid] = labels
    return full_labels


In [ ]:

def adaptive_train_test_split(df: pd.DataFrame, target_col: str,
                              test_size: float = 0.1, min_per_bin: int = 3,
                              random_state: int = 42):
    """
    Try stratified train/test splits with decreasing bin counts (10 bins down to 2).
    Ensures each bin has >= min_per_bin samples and appears in both train and test.
    Falls back to random split if no stratification is feasible.
    """
    for bins in range(10, 1, -1):
        strat_labels = build_strat_labels(df[target_col], n_bins=bins, min_per_bin=min_per_bin)
        if strat_labels is None:
            continue
        train_df, test_df = train_test_split(df, test_size=test_size, stratify=strat_labels,
                                             shuffle=True, random_state=random_state)
        # Verify that all bins are represented in both splits
        train_bins = set(strat_labels[train_df.index])
        test_bins = set(strat_labels[test_df.index])
        if train_bins == test_bins and (-1 not in train_bins):
            print(f"✔ Stratified split succeeded with {bins} bins.")
            return train_df, test_df
    # If we exit the loop, no stratified scheme was viable
    print("⚠ Stratified split infeasible – using a random split.")
    return train_test_split(df, test_size=test_size, shuffle=True, random_state=random_state)

# Perform train/test split
train_df, test_df = adaptive_train_test_split(model_data, TARGET_COL, test_size=0.10,
                                             min_per_bin=3, random_state=RANDOM_SEED)


✔ Stratified split succeeded with 10 bins.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ─── Global font config ────
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Nimbus Roman", "Liberation Serif", "Times", "Tinos"],
    "font.size": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "axes.titlesize": 16,
})

def save_distribution_plot(
    df: pd.DataFrame,
    column: str,
    filename: str,
    base_interval: float = 0.2,
    splits_per_interval: int = 10
) -> None:
    """Histogram of *column* with consistent 14-pt serif font."""
    tick_step = base_interval / splits_per_interval
    dmin, dmax = df[column].min(), df[column].max()
    start = np.floor(dmin / tick_step) * tick_step
    stop  = np.ceil (dmax / tick_step) * tick_step
    bin_edges = np.arange(start, stop + tick_step, tick_step)

    fig, ax = plt.subplots(figsize=(8, 6))
    df[column].hist(bins=bin_edges, color="skyblue",
                    edgecolor="skyblue", alpha=0.75, ax=ax)

    for side in ("top", "bottom", "left", "right"):
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("black")

    ax.set_xlabel("Target Variable Values")
    ax.set_ylabel("Frequency")

    ax.set_xticks(np.arange(start, stop + tick_step, base_interval))
    ax.set_xticks(np.arange(start, stop + tick_step, tick_step), minor=True)
    ax.tick_params(axis="both", which="both", labelsize=14)

    ax.grid(False)
    plt.tight_layout()
    plt.savefig(filename, dpi=600, bbox_inches="tight")
    plt.savefig(filename.rsplit(".", 1)[0] + ".pdf", format="pdf",
                bbox_inches="tight")
    plt.close()

# ─── Create the three histograms ────────────────────────────────────────────
save_distribution_plot(model_data, TARGET_COL, "full_target_distribution.png")
save_distribution_plot(train_df,  TARGET_COL, "train_target_distribution.png")
save_distribution_plot(test_df,   TARGET_COL, "test_target_distribution.png")
print("✔ Target-distribution plots saved (PNG + PDF, uniform 14-pt serif fonts).")

# Build feature matrices
X_train = train_df[input_features];  y_train = train_df[TARGET_COL]
X_test  = test_df[input_features];   y_test  = test_df[TARGET_COL]
print(f"→ Training: {len(X_train):,d} rows | Test: {len(X_test):,d} rows")


In [ ]:
# 5-fold cross-validation setup (for evaluating hyperparameter performance)
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
def cv_rmse(estimator, X, y):
    """Return mean 5-fold RMSE for a given estimator (positive value)."""
    rmse_scores = cross_val_score(estimator, X, y,
                                  scoring='neg_root_mean_squared_error',
                                  cv=cv, n_jobs=-1)
    return -rmse_scores.mean()


In [ ]:


# ─────────────────────────────────────────────────────────────────────────────
#   2‑PHASE OPTUNA TUNING
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np, pandas as pd, optuna, lightgbm as lgb
from math import sqrt
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

cv5 = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# ─────────────────────────────────────────────────────────────────────────────
# Helper: 5‑fold CV RMSE *with Optuna pruning support*
# ─────────────────────────────────────────────────────────────────────────────
def cv_rmse_with_reporting(model, X, y, trial):
    """
    X : pandas.DataFrame
    y : pandas.Series OR numpy array
    """
    y_arr = y.to_numpy() if isinstance(y, pd.Series) else np.asarray(y)
    fold_rmses = []

    for fold_idx, (tr_idx, va_idx) in enumerate(cv5.split(X, y_arr)):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]

        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)
        rmse = sqrt(mean_squared_error(y_va, preds))
        fold_rmses.append(rmse)

        # Let Optuna's pruner observe the intermediate result
        trial.report(rmse, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_rmses))

# ─────────────────────────────────────────────────────────────────────────────
#  PHASE‑1  (broad search)
# ─────────────────────────────────────────────────────────────────────────────
def objective_phase1(trial):
    params = {
        "boosting_type":  "gbdt",
        "objective":      "regression",
        "random_state":   RANDOM_SEED,
        "verbosity":      -1,
        "n_jobs":         -1,

        # ─ hyper‑parameters ─
        "learning_rate":   trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        "num_leaves":      trial.suggest_int  ("num_leaves",    16,   256, log=True),
        "max_depth":       trial.suggest_int  ("max_depth",     3,    30),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample":       trial.suggest_float("subsample",     0.5,  1.0),
        "bagging_freq":    1,
        "colsample_bytree":trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "reg_alpha":       trial.suggest_float("reg_alpha",     1e-5, 10.0, log=True),
        "reg_lambda":      trial.suggest_float("reg_lambda",    1e-5, 10.0, log=True),
        "n_estimators":    trial.suggest_int  ("n_estimators",  100,  2000),
    }
    model = lgb.LGBMRegressor(**params)
    return cv_rmse_with_reporting(model, X_train, y_train, trial)

study1 = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2),
)
study1.optimize(objective_phase1, n_trials=150)
print(f"Phase‑1 best RMSE: {study1.best_value:.5f}")
best1 = study1.best_params

# ─────────────────────────────────────────────────────────────────────────────
#  Range‑narrow helpers
# ─────────────────────────────────────────────────────────────────────────────
def narrow_float(name, lo, hi, pct=0.25, log=False):
    best = best1[name]
    new_lo = max(lo, best * (1 - pct))
    new_hi = min(hi, best * (1 + pct))
    if np.isclose(new_lo, new_hi):
        new_lo, new_hi = lo, hi
    return (new_lo, new_hi)

def narrow_int(name, lo, hi, pct=0.25):
    best = best1[name]
    span = int(max(2, round(best * pct)))
    new_lo = max(lo, best - span)
    new_hi = min(hi, best + span)
    if new_lo == new_hi:
        new_lo, new_hi = lo, hi
    return (new_lo, new_hi)

# ─────────────────────────────────────────────────────────────────────────────
#  PHASE‑2  (refined search)
# ─────────────────────────────────────────────────────────────────────────────
def objective_phase2(trial):
    params = {
        "boosting_type": "gbdt",
        "objective":     "regression",
        "random_state":  RANDOM_SEED,
        "verbosity":     -1,
        "n_jobs":        -1,

        "learning_rate": trial.suggest_float(
            "learning_rate", *narrow_float("learning_rate", 1e-3, 0.2, pct=0.25), log=True
        ),
        "num_leaves": trial.suggest_int(
            "num_leaves", *narrow_int("num_leaves", 16, 256, pct=0.25), log=True
        ),
        "max_depth": trial.suggest_int("max_depth", *narrow_int("max_depth", 3, 30, pct=0.25)),
        "min_child_samples": trial.suggest_int(
            "min_child_samples", *narrow_int("min_child_samples", 5, 50, pct=0.25)
        ),
        "subsample": trial.suggest_float("subsample", *narrow_float("subsample", 0.5, 1.0, pct=0.2)),
        "bagging_freq": 1,
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", *narrow_float("colsample_bytree", 0.3, 1.0, pct=0.2)
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", *narrow_float("reg_alpha", 1e-5, 10.0, pct=0.5), log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", *narrow_float("reg_lambda", 1e-5, 10.0, pct=0.5), log=True
        ),
        "n_estimators": trial.suggest_int(
            "n_estimators", *narrow_int("n_estimators", 100, 2000, pct=0.25)
        ),
    }
    model = lgb.LGBMRegressor(**params)
    return cv_rmse_with_reporting(model, X_train, y_train, trial)

class NoImprovementStopper:
    def __init__(self, patience=40, min_delta=1e-4):
        self.patience, self.min_delta = patience, min_delta
        self.best_val, self.bad = np.inf, 0
    def __call__(self, study, trial):
        if study.best_value < self.best_val - self.min_delta:
            self.best_val, self.bad = study.best_value, 0
        else:
            self.bad += 1
        if self.bad >= self.patience:
            print(f"\n★ Early‑stopping Phase‑2 after {self.patience} stagnant trials ★")
            study.stop()

study2 = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2),
)
study2.optimize(
    objective_phase2,
    n_trials=150,
    callbacks=[NoImprovementStopper(patience=40, min_delta=1e-4)],
)

# ─────────────────────────────────────────────────────────────────────────────
#  Pick the overall best study
# ─────────────────────────────────────────────────────────────────────────────
study_lgb = study2 if study2.best_value < study1.best_value else study1
print(f"\n✔  Final best RMSE: {study_lgb.best_value:.5f}")
print("Best hyper‑parameters:\n", study_lgb.best_params)


[I 2025-07-03 09:13:05,082] A new study created in memory with name: no-name-cb192d64-b3e2-4e1c-ba5a-dd253cdfed7b
[I 2025-07-03 09:14:55,185] Trial 0 finished with value: 0.15229776429470673 and parameters: {'learning_rate': 0.00727491708802781, 'num_leaves': 223, 'max_depth': 23, 'min_child_samples': 32, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.40919616423534183, 'reg_alpha': 2.231010801867923e-05, 'reg_lambda': 1.574189004745663, 'n_estimators': 1242}. Best is trial 0 with value: 0.15229776429470673.
[I 2025-07-03 09:15:12,774] Trial 1 finished with value: 0.15480366516424046 and parameters: {'learning_rate': 0.04258888210290081, 'num_leaves': 16, 'max_depth': 30, 'min_child_samples': 43, 'subsample': 0.6061695553391381, 'colsample_bytree': 0.42727747704497043, 'reg_alpha': 0.00012601639723276795, 'reg_lambda': 0.0006690421166498799, 'n_estimators': 1097}. Best is trial 0 with value: 0.15229776429470673.
[I 2025-07-03 09:15:31,515] Trial 2 finished with value: 0.1520187

Phase‑1 best RMSE: 0.15133


[I 2025-07-03 11:31:00,686] Trial 0 finished with value: 0.1516075595506605 and parameters: {'learning_rate': 0.003096315966868993, 'num_leaves': 142, 'max_depth': 28, 'min_child_samples': 24, 'subsample': 0.6184997717884679, 'colsample_bytree': 0.4654888720156558, 'reg_alpha': 1.0636583010549368e-05, 'reg_lambda': 0.05693072284049491, 'n_estimators': 1243}. Best is trial 0 with value: 0.1516075595506605.
[I 2025-07-03 11:32:06,265] Trial 1 finished with value: 0.151423637241718 and parameters: {'learning_rate': 0.003671461946867317, 'num_leaves': 87, 'max_depth': 30, 'min_child_samples': 27, 'subsample': 0.6346564957306773, 'colsample_bytree': 0.4710657811701928, 'reg_alpha': 1.215151080498706e-05, 'reg_lambda': 0.03070690810034198, 'n_estimators': 1198}. Best is trial 1 with value: 0.151423637241718.
[I 2025-07-03 11:33:09,365] Trial 2 finished with value: 0.15174215618164816 and parameters: {'learning_rate': 0.003188456362404695, 'num_leaves': 101, 'max_depth': 26, 'min_child_sample


★ Early‑stopping Phase‑2 after 40 stagnant trials ★

✔  Final best RMSE: 0.15132
Best hyper‑parameters:
 {'learning_rate': 0.0037287937507853104, 'num_leaves': 128, 'max_depth': 20, 'min_child_samples': 20, 'subsample': 0.6983468372166726, 'colsample_bytree': 0.5894375457971144, 'reg_alpha': 1.931426037269914e-05, 'reg_lambda': 0.03514471353322322, 'n_estimators': 1294}


In [ ]:

# Retrieve the best parameters and update with static parameters
best_params_lgb = study_lgb.best_params.copy()
best_params_lgb.update({
    'boosting_type': 'gbdt',
    'objective': 'regression',
    'random_state': RANDOM_SEED,
    'verbosity': -1,
    'n_jobs': -1,
    'bagging_freq': 1  # ensure subsample (bagging_fraction) is actually used
})

# Initialize and train the final LightGBM model on the full training data
model_lgb = lgb.LGBMRegressor(**best_params_lgb)
model_lgb.fit(X_train, y_train)
print("✔ Final LightGBM model trained with best hyperparameters.")


✔ Final LightGBM model trained with best hyperparameters.
